# KVTC — KV Cache Transform Coding Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OnlyTerp/kvtc/blob/master/notebooks/demo.ipynb)
[![GitHub](https://img.shields.io/github/stars/OnlyTerp/kvtc?style=social)](https://github.com/OnlyTerp/kvtc)

First open-source implementation of [NVIDIA's KVTC](https://arxiv.org/abs/2511.01815) (ICLR 2026).

**What this does:**
1. Loads TinyLlama-1.1B (fits in Colab free tier)
2. Calibrates PCA bases from 10 sample texts (~2 seconds)
3. Compresses KV cache at 3 quality levels (8x, 16x, 32x)
4. Measures cosine similarity between original and reconstructed KV

In [ ]:
!pip install -q torch transformers
!pip install -q git+https://github.com/OnlyTerp/kvtc.git

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.pca import PCACalibrator
from src.pipeline import KVTCCompressor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Load Model

In [ ]:
model_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
)
if not torch.cuda.is_available():
    model = model.to(device)
model.eval()
print(f'Loaded {model_name}')
print(f'Layers: {model.config.num_hidden_layers}, KV heads: {model.config.num_key_value_heads}')

In [ ]:
def extract_kv(past_kv):
    """Handle all HuggingFace cache formats."""
    if hasattr(past_kv, 'layers'):
        return [(l.keys, l.values) for l in past_kv.layers]
    if hasattr(past_kv, 'key_cache'):
        return [(past_kv.key_cache[i], past_kv.value_cache[i]) for i in range(len(past_kv.key_cache))]
    return [(item[0], item[1]) for item in past_kv]

## Step 1: Calibrate PCA

In [ ]:
import time

cal_texts = [
    'The quick brown fox jumps over the lazy dog in a beautiful meadow.',
    'Machine learning models require careful optimization and tuning.',
    'KV cache compression unlocks longer contexts for LLM inference.',
    'Dynamic programming finds optimal bit allocations efficiently.',
    'Principal component analysis decorrelates feature dimensions.',
    'Entropy coding exploits statistical redundancy in quantized data.',
    'Attention mechanisms allow models to focus on relevant tokens.',
    'GPU memory management is critical for serving large models.',
    'Neural network architectures continue to evolve rapidly.',
    'The transformer architecture has revolutionized NLP since 2017.',
]

# Detect head_dim
with torch.no_grad():
    test = tokenizer('test', return_tensors='pt').to(device)
    kv = extract_kv(model(**test, use_cache=True).past_key_values)
    head_dim = kv[0][0].shape[-1]
    n_layers = len(kv)
print(f'head_dim={head_dim}, layers={n_layers}')

# Calibrate
calibrator = PCACalibrator(head_group_size=1)
t0 = time.time()
for text in cal_texts:
    inputs = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(**inputs, use_cache=True)
    positions = torch.arange(inputs['input_ids'].shape[1], device='cpu')
    for li, (k, v) in enumerate(extract_kv(out.past_key_values)):
        calibrator.collect(li, 'keys', k[0].transpose(0,1).cpu().float(), positions)
        calibrator.collect(li, 'values', v[0].transpose(0,1).cpu().float())
print(f'Calibration done in {time.time()-t0:.1f}s')

## Step 2: Compress at Different Quality Levels

In [ ]:
# Long prompt to have a middle region (>132 tokens)
prompt = (
    'Explain the concept of attention mechanisms in neural networks. '
    'Cover the history starting from Bahdanau attention in 2014, '
    'then move to the transformer architecture introduced in 2017. '
    'Discuss multi-head attention, self-attention, cross-attention, '
    'and how they differ. Explain the key, query, and value projections '
    'in detail. Discuss positional encodings and why they are needed. '
    'Cover the softmax operation and how it creates attention weights. '
    'Explain masked attention for autoregressive generation. '
    'Discuss grouped query attention and multi-query attention optimizations. '
    'Cover flash attention and memory efficient attention implementations.'
)

inputs = tokenizer(prompt, return_tensors='pt').to(device)
seq_len = inputs['input_ids'].shape[1]
print(f'Prompt tokens: {seq_len}')

with torch.no_grad():
    out = model(**inputs, use_cache=True)
positions = torch.arange(seq_len, device='cpu')

# Build KV cache tensor
all_keys, all_values = [], []
for k, v in extract_kv(out.past_key_values):
    all_keys.append(k[0].transpose(0,1).cpu().float())
    all_values.append(v[0].transpose(0,1).cpu().float())
kv_cache = {'keys': torch.stack(all_keys), 'values': torch.stack(all_values)}

modes = [
    {'name': 'High Quality', 'ratio': 0.25},
    {'name': 'Balanced', 'ratio': 0.12},
    {'name': 'Aggressive', 'ratio': 0.06},
]

print(f'\n{"Mode":<16} {"Compression":>12} {"Key Cosine":>12} {"Value Cosine":>14}')
print('-' * 58)

for mode in modes:
    cal = calibrator.compute(bit_budget_ratio=mode['ratio'])
    compressor = KVTCCompressor(cal)
    compressed = compressor.compress(kv_cache, positions)
    restored = compressor.decompress(compressed)
    
    key_cos = F.cosine_similarity(
        kv_cache['keys'].reshape(1,-1), restored['keys'].reshape(1,-1)
    ).item()
    val_cos = F.cosine_similarity(
        kv_cache['values'].reshape(1,-1), restored['values'].reshape(1,-1)
    ).item()
    cr = compressed.metadata.compression_ratio
    
    print(f'{mode["name"]:<16} {cr:>11.1f}x {key_cos:>12.4f} {val_cos:>14.4f}')

## What's Happening Under the Hood

```
KV Cache → [Undo RoPE] → [PCA Transform] → [DP Quantize] → [DEFLATE] → Compressed
                                                                            │
Restored ← [Reapply RoPE] ← [PCA Inverse] ← [Dequantize] ← [Inflate] ←────┘
```

- **Attention sinks** (first 4 tokens) and **sliding window** (last 128) are never compressed
- PCA eigenvectors computed once during calibration, reused for all prompts
- DP assigns 0 bits to low-variance components → free dimensionality reduction

---

**Paper:** [KVTC: KV Cache Transform Coding](https://arxiv.org/abs/2511.01815) (ICLR 2026, NVIDIA)

**GitHub:** [OnlyTerp/kvtc](https://github.com/OnlyTerp/kvtc)